# Прототип книжного рекомендательного сервиса на Goodbooks-10k

Код моделей и метрик вынесен в пакет `src/recsys`, ноутбук содержит анализ, эксперименты и выводы. Данные загружаются с Kaggle через kagglehub.

Запуск в Google Colab: выполнить в первой ячейке `!git clone https://github.com/EgorCOStylev/goodbooks-recsys.git` и `%cd goodbooks-recsys`, затем `!pip install -q scikit-surprise`. Локально: `pip install -r requirements.txt` и запуск из папки notebooks.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from scipy import sparse
from sklearn.preprocessing import normalize

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from recsys import data, popularity, content, item_cf, metrics
from recsys.plots import show_table

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

DATA_DIR = Path(kagglehub.dataset_download('zygmunt/goodbooks-10k'))
SEED     = 42

## 1. Загрузка данных

Датасет загружается с Kaggle (zygmunt/goodbooks-10k). Это первая версия Goodbooks-10k, та же, что в материалах курса: ratings.csv содержит 981 756 оценок с колонками book_id, user_id, rating и без временных меток. Файлы to_read.csv и sample_book.xml в работе не используются.

In [ ]:
raw       = data.load_raw(DATA_DIR)
ratings   = raw['ratings']
books     = raw['books']
tags      = raw['tags']
book_tags = raw['book_tags']

ratings.head()

In [ ]:
for name, df in [('ratings', ratings), ('books', books), ('tags', tags), ('book_tags', book_tags)]:
    print(f'{name:<10} строк: {df.shape[0]:>9,}  колонок: {df.shape[1]}')
print(f'\nПользователей: {ratings.user_id.nunique():,}, книг в оценках: {ratings.book_id.nunique():,}')

## 2. Очистка данных

В ratings.csv встречаются повторные пары пользователь–книга. Если их не убрать, одна и та же пара может попасть и в обучающую, и в тестовую выборку. Повторы схлопываются в одну запись со средней оценкой.

book_tags.csv ссылается на книги через goodreads_book_id, а ratings.csv через id из books.csv. Это разные идентификаторы, поэтому теги переводятся на id через books.csv.

У 585 книг нет original_title, для них берётся title.

In [ ]:
ratings, dup_rows, conflicts = data.dedup_ratings(ratings)
books['original_title']      = books.original_title.fillna(books.title)
book_tags                    = data.attach_tags(book_tags, books, tags)

book_tags.head()

In [ ]:
print(f'Строк в повторных парах: {dup_rows:,}, из них пар с разными оценками: {conflicts:,}')
print(f'Оценок после очистки: {len(ratings):,}')
print(f'Книг с тегами: {book_tags.book_id.nunique():,}, связей книга–тег: {len(book_tags):,}')

## 3. Распределение оценок

In [ ]:
rating_share = ratings.rating.round().astype(int).value_counts(normalize=True).sort_index()
rating_share.to_frame('доля')

In [ ]:
print(f'Средняя оценка: {ratings.rating.mean():.2f}')
print(f'Доля оценок >= 4: {(ratings.rating >= 4).mean():.1%}')
print(f'Доля оценок >= 5: {(ratings.rating >= 5).mean():.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(rating_share.index, rating_share.values, color='steelblue')
ax.axvline(ratings.rating.mean(), color='coral', lw=2, ls='--', label=f'среднее {ratings.rating.mean():.2f}')
ax.set_xlabel('Оценка')
ax.set_ylabel('Доля оценок')
ax.set_title('Распределение оценок')
ax.legend()
plt.tight_layout(); plt.show()

Оценки смещены вверх: 66,2% оценок равны 4 или 5, средняя 3,86, единицу ставят в 2% случаев. Пользователи оценивают в основном книги, которые сами выбрали и дочитали, поэтому низкие оценки редки. Известные оценки не являются случайной выборкой из всех пар пользователь–книга (missing not at random), и предсказание оценки для непрочитанной книги переносит это смещение.

Для метрик ранжирования это значит, что порог релевантности 4 отделяет только треть оценок, и релевантной окажется большая часть прочитанного. Поэтому в разделе с метриками основной порог 4 дополнен проверкой с порогом 5, при котором релевантны 29,8% оценок.

## 4. Активность пользователей

In [ ]:
user_counts = ratings.groupby('user_id').size()
user_counts.describe().to_frame('оценок на пользователя').T

In [ ]:
for n in (3, 5, 10):
    print(f'Пользователей с <= {n:>2} оценками: {(user_counts <= n).mean():.1%}')
top_share = user_counts.sort_values(ascending=False)
top_share = top_share.head(int(len(top_share) * 0.1)).sum() / top_share.sum()
print(f'Доля оценок от 10% самых активных пользователей: {top_share:.1%}')

In [ ]:
users_per_count = user_counts.value_counts().sort_index()
lorenz_users    = np.arange(1, len(user_counts) + 1) / len(user_counts)
lorenz_ratings  = np.cumsum(np.sort(user_counts.values)[::-1]) / user_counts.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(users_per_count.index, users_per_count.values, color='steelblue', lw=2)
axes[0].set_yscale('log')
axes[0].set_xlabel('Количество оценок у пользователя')
axes[0].set_ylabel('Количество пользователей (лог. шкала)')
axes[0].set_title('Пользователи и количество их оценок')

axes[1].plot(lorenz_users, lorenz_ratings, color='steelblue', lw=2, label='фактическое')
axes[1].plot([0, 1], [0, 1], color='coral', lw=2, ls='--', label='равномерное')
axes[1].set_xlabel('Доля пользователей (от самых активных)')
axes[1].set_ylabel('Накопленная доля оценок')
axes[1].set_title('Концентрация оценок по пользователям')
axes[1].legend()
plt.tight_layout(); plt.show()

Медиана 8 оценок на пользователя, у 25,8% пользователей не больше 3 оценок, у 57% не больше 10. На логарифмической шкале распределение близко к прямой, то есть число пользователей убывает почти экспоненциально с ростом активности. 10% самых активных пользователей дают 44,9% оценок, и модели, обученные на этих данных, подстраиваются прежде всего под их вкусы.

Для большинства пользователей истории почти нет, это холодный старт: коллаборативным моделям не с чем сравнивать пользователя с тремя оценками, и для таких пользователей нужны популярные книги или контентная модель. Максимум ровно 200 оценок и подъём у правого края графика похожи на ограничение, наложенное при формировании выборки.

## 5. Популярность книг

Количество оценок книги можно взять из двух источников: из ratings.csv (выборка, на которой обучаются модели) и из ratings_count в books.csv (все оценки книги на Goodreads).

In [ ]:
book_counts = ratings.groupby('book_id').size()
pd.DataFrame({'в ratings.csv': book_counts.describe(),
              'ratings_count на Goodreads': books.ratings_count.describe()}).T

In [ ]:
gr_sorted = books.ratings_count.sort_values(ascending=False).values
print(f'Книг ровно со 100 оценками в ratings.csv: {(book_counts == 100).mean():.1%}')
print(f'Доля оценок Goodreads у топ-1% книг: {gr_sorted[:100].sum() / gr_sorted.sum():.1%}')
print(f'Доля оценок Goodreads у топ-10% книг: {gr_sorted[:1000].sum() / gr_sorted.sum():.1%}')
print(f'Максимум / медиана ratings_count: {gr_sorted.max() / np.median(gr_sorted):.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(book_counts, bins=50, color='steelblue')
axes[0].set_xlabel('Количество оценок книги в ratings.csv')
axes[0].set_ylabel('Количество книг')
axes[0].set_title('ratings.csv: не больше 100 оценок на книгу')

axes[1].plot(np.arange(1, len(gr_sorted) + 1), gr_sorted, color='steelblue', lw=2)
axes[1].axvline(1000, color='coral', lw=2, ls='--', label='граница топ-10%')
axes[1].set_yscale('log')
axes[1].set_xlabel('Ранг книги по популярности')
axes[1].set_ylabel('ratings_count на Goodreads (лог. шкала)')
axes[1].set_title('Длинный хвост популярности на Goodreads')
axes[1].legend()
plt.tight_layout(); plt.show()

В ratings.csv у 62,5% книг ровно 100 оценок, у 94% не меньше 90. Авторы выборки ограничили число оценок на книгу, поэтому длинного хвоста по числу оценок в ratings.csv нет, и число оценок из ratings.csv почти не различает книги по популярности.

Реальное смещение популярности видно по ratings_count на Goodreads: топ-1% книг собирает 23,4% всех оценок, топ-10% собирает 57,1%, самая популярная книга оценена в 226 раз чаще медианной. В ratings.csv это смещение спрятано ограничением в 100 оценок, но оно есть в самом каталоге и в том, какие книги пользователи вообще выбирали читать. Неперсонализированная модель на следующем этапе ранжирует по сглаженной средней оценке, как требует задание, а ratings_count используется для анализа смещения популярности в рекомендациях.

## 6. Теги книг

Большая часть самых частых тегов описывает полку пользователя, а не книгу: to-read, favorites, owned, kindle. Тег считается служебным, если он состоит только из служебных слов и чисел.

In [ ]:
tag_books = book_tags.groupby('tag_name').book_id.nunique().sort_values(ascending=False)
service   = np.array([content.is_service(t) for t in tag_books.index])

tag_books.head(10).to_frame('книг с тегом')

In [ ]:
print(f'Уникальных тегов: {len(tag_books):,}, служебных: {service.sum():,}')
print(f'Служебных среди топ-30 по числу книг: {service[:30].sum()}')

In [ ]:
content_top = tag_books[~service].head(20)
service_top = tag_books[service].head(20)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(service_top.index[::-1], service_top.values[::-1], color='coral')
axes[0].set_xlabel('Количество книг с тегом')
axes[0].set_title('Топ-20 служебных тегов')

axes[1].barh(content_top.index[::-1], content_top.values[::-1], color='steelblue')
axes[1].set_xlabel('Количество книг с тегом')
axes[1].set_title('Топ-20 содержательных тегов')
plt.tight_layout(); plt.show()

27 из 30 самых частых тегов служебные: они описывают полку пользователя (to-read, owned, kindle), а не содержание книги. Тег to-read стоит у 9 983 книг из 10 000. TF-IDF сам опустит вес таких тегов почти до нуля, но служебных тегов с высокой частотой много, и вместе они размывают текстовые профили книг. Поэтому для контентной модели служебные теги удаляются заранее. Среди содержательных тегов лидируют широкие жанры: fiction, fantasy, romance, mystery, young-adult.

## 7. Разреженность матрицы взаимодействий

In [ ]:
idx = data.Index(users=np.sort(ratings.user_id.unique()), books=np.sort(books.id.unique()))
R   = idx.to_csr(ratings)

dense_gb  = R.shape[0] * R.shape[1] * 8 / 1024**3
sparse_mb = (R.data.nbytes + R.indices.nbytes + R.indptr.nbytes) / 1024**2
print(f'Матрица: {R.shape[0]:,} x {R.shape[1]:,}, ненулевых: {R.nnz:,}')
print(f'Заполненность: {R.nnz / (R.shape[0] * R.shape[1]):.3%}')
print(f'Память: плотная {dense_gb:.1f} ГБ, CSR {sparse_mb:.1f} МБ')

Заполнено 0,183% матрицы. Плотная матрица float64 заняла бы 4 ГБ, формат CSR занимает 11,4 МБ, поэтому дальше все вычисления идут на разреженных матрицах.

## 8. Проблемы данных по итогам EDA

Разреженность: заполнено 0,18% матрицы, у двух случайных пользователей с медианной активностью почти нет общих книг. Сходство пользователей по общим оценкам ненадёжно, поэтому коллаборативная часть строится на сходстве книг: у каждой книги около 100 оценок, и пересечения между книгами находятся чаще.

Холодный старт: у четверти пользователей не больше 3 оценок. Для них персональные модели будут работать хуже популярных книг, и при оценке качества нужно отдельно смотреть, как модели ведут себя на малоактивных пользователях.

Смещение популярности: в каталоге оно сильное (топ-10% книг собирают 57% оценок Goodreads), но в ratings.csv скрыто ограничением в 100 оценок на книгу. Модель, которая хорошо выглядит на этих данных, в реальном сервисе с неограниченной популярностью может тянуться к хитам сильнее.

Смещение оценок вверх: две трети оценок не ниже 4, что делает порог релевантности решением, которое нужно обосновывать.

Технические особенности: повторные пары пользователь–книга, разные идентификаторы книг в ratings.csv и book_tags.csv и служебные теги. Временных меток нет, поэтому разделение на обучающую и тестовую выборки делается случайно внутри каждого пользователя.

## 9. Неперсонализированная модель

Сортировка по средней оценке нестабильна: книга с тремя пятёрками обгонит книгу с сотней оценок 4,8. Поэтому книги с числом оценок меньше порога отбрасываются, а средняя оценка сглаживается к общему среднему (байесовское среднее): score = (v·R + m·C) / (v + m), где v число оценок книги, R её средняя оценка, C общее среднее, m сила сглаживания. Пока у книги мало оценок, её score близок к C, с ростом v он приближается к R.

Функция принимает таблицу оценок аргументом, чтобы на этапе сравнения моделей её можно было обучить только на обучающей выборке.

In [ ]:
pop_top = popularity.top_popular(ratings, n=10)
pop_top.to_frame('score').join(books.set_index('id')[['title', 'authors', 'ratings_count']])

In [ ]:
stats = ratings.groupby('book_id').rating.agg(['mean', 'count'])
print(f'Общее среднее C: {ratings.rating.mean():.3f}, m (медиана числа оценок): {stats["count"].median():.0f}')
print(f'Книг после порога 50 оценок: {(stats["count"] >= 50).sum():,} из {len(stats):,}')
print(f'Медианный ratings_count на Goodreads у топ-10: {books.set_index("id").loc[pop_top.index, "ratings_count"].median():,.0f}')
print(f'Медианный ratings_count по каталогу: {books.ratings_count.median():,.0f}')

Порог в 50 оценок отсекает 10 книг из 10 000: в ratings.csv почти у всех книг около 100 оценок, и сглаживание при m = 100 сдвигает оценки всех книг примерно одинаково. Поэтому порядок определяет просто средняя оценка.

В топ-10 семь сборников комиксов Calvin and Hobbes, подарочный набор Гарри Поттера, учебная Библия и вторая книга серии A Court of Thorns and Roses. Высокую среднюю оценку здесь дают читатели, которые отобрали себя сами: сборник комиксов покупают поклонники комикса, продолжение серии читают те, кому понравилась первая книга. Медианный ratings_count у топ-10 на Goodreads 17 704, у каталога в целом 21 156, то есть «лучшие» книги известны меньше средней книги.

Для пользователя без истории такая выдача неудачна: это книги для тех, кто уже в теме. Рекомендация по числу оценок из ratings.csv невозможна (у всех около 100), поэтому популярность как известность есть только в ratings_count Goodreads, которого нет в ratings.csv. Модель по средней оценке остаётся бейзлайном, как требует задание.

## 10. Контентная модель

Текстовый профиль книги собирается из названия (original_title) и содержательных тегов. Служебные теги удаляются фильтром из раздела 6, из оставшихся берутся 20 самых частых для книги. Тег записывается одним токеном (young-adult превращается в young_adult), чтобы TF-IDF не разбивал его на слова и не смешивал young_adult с отдельным тегом adult. Слова названия идут отдельными токенами, английские стоп-слова удаляются.

In [ ]:
profiles = content.build_profiles(books, book_tags, top_tags=20)
profiles.head()

In [ ]:
tfidf, X = content.fit_tfidf(profiles)
book_ids = profiles.index.values

print(f'Матрица TF-IDF: {X.shape[0]:,} книг x {X.shape[1]:,} токенов, ненулевых: {X.nnz:,}')
print(f'Средняя длина профиля: {profiles.str.split().str.len().mean():.1f} токенов')

Строки TF-IDF нормированы по L2, поэтому косинусная близость сводится к скалярному произведению. Сходство одной книги со всеми остальными считается одним умножением разреженной строки на транспонированную матрицу, без цикла по книгам.

In [ ]:
def get_similar_books(book_id, N=5):
    sims = content.similar_books(X, book_ids, book_id, n=N)
    return books.set_index('id').loc[sims.index, ['title', 'authors']].assign(similarity=sims.round(3).values)

get_similar_books(2)

In [ ]:
for bid in (1, 5, 33):
    print(books.set_index('id').loc[bid, 'title'])
    print(get_similar_books(bid).to_string(), end='\n\n')

Для книг из серий модель находит другие книги той же серии и сопутствующие издания: для «Гарри Поттера» все пять соседей это следующие тома, для «Голодных игр» подарочный набор, продолжения и путеводители по трилогии. Формально это верное сходство, по названиям и тегам книги почти совпадают, но как рекомендация оно бесполезно: читатель первой книги серии знает о продолжениях без подсказок. Название приносит общие токены вроде hunger и games, которые перевешивают жанровые теги.

Там, где серии нет, модель работает осмысленно. Для «Великого Гэтсби» она предлагает американскую классику из школьной программы (Steinbeck, Salinger, Harper Lee), для «Мемуаров гейши» книги о Японии и японских авторов. Это контентное сходство без единой оценки, и именно поэтому модель пригодна для новых книг, у которых ещё нет истории. Её слабость видна там же: она повторяет то, что пользователь уже выбрал, и не выводит за пределы привычного жанра.

## 11. Разделение на обучающую и тестовую выборки

Временных меток нет, поэтому разделение делается случайно внутри каждого пользователя: у каждого пользователя с не менее чем 5 оценками 20% оценок уходят в тест. Пользователи с меньшим числом оценок целиком остаются в обучающей выборке: их оценки помогают считать сходство книг, но проверять на них рекомендации не на чем. Так каждый тестовый пользователь есть в обучении, и тест проверяет предсказание новых для него книг. Все модели дальше обучаются только на train.

In [ ]:
TEST_SHARE       = 0.2
MIN_USER_RATINGS = 5

train, test = data.split_per_user(ratings, test_share=TEST_SHARE, min_user_ratings=MIN_USER_RATINGS, seed=SEED)

train.head()

In [ ]:
print(f'train: {len(train):,} оценок, test: {len(test):,} оценок ({len(test) / len(ratings):.1%})')
print(f'Пользователей в тесте: {test.user_id.nunique():,} из {ratings.user_id.nunique():,}')
print(f'Тестовых пользователей без оценок в train: {(~test.user_id.isin(train.user_id)).sum()}')
print(f'Книг из теста, которых нет в train: {(~test.book_id.isin(train.book_id)).sum()}')

## 12. Item-based коллаборативная фильтрация

Сходство книг считается скорректированным косинусом: из каждой оценки вычитается средняя оценка пользователя, после чего считается косинус между столбцами книг. Центрирование по пользователю убирает разницу шкал: строгий читатель ставит любимой книге 4, щедрый ставит 5, и без центрирования это выглядело бы как разный вкус.

У большинства пар книг общих читателей единицы, и косинус по двум-трём общим оценкам случаен. Поэтому сходство сжимается по числу общих читателей n: sim · n / (n + λ) при λ = 10. Для каждой книги хранятся только k ближайших соседей с положительным сходством, поэтому матрица сходства остаётся разреженной.

Предсказание оценки пользователя u для книги i: r̂ = r̄_u + Σ s_ij · (r_uj − r̄_u) / (Σ s_ij + 1), где суммы идут по соседям книги i, которые пользователь оценил. Единица в знаменателе сжимает предсказание к средней оценке пользователя, когда опора на соседей слабая: одна книга с близостью 0,05 почти не сдвигает прогноз.

In [ ]:
R_train                = idx.to_csr(train)
R_dev, R_ind, user_mean = item_cf.center_by_user(R_train)
n_rated                = np.diff(R_train.indptr)

pd.Series(user_mean).describe().to_frame('средняя оценка пользователя в train').T

In [ ]:
S = item_cf.item_similarity(R_dev, R_ind, k_neighbors=50, shrink=10)
print(f'Матрица сходства: {S.shape[0]:,} x {S.shape[1]:,}, ненулевых: {S.nnz:,} ({S.nnz / S.shape[0]:.1f} соседей на книгу)')

In [ ]:
def predict_rating(user_id, book_id):
    u = idx.user_pos.get_loc(user_id)
    return item_cf.explicit_scores(R_dev, R_ind, user_mean, S, [u])[0, idx.book_pos.get_loc(book_id)]

def recommend_item_cf(user_id, N=10):
    u   = idx.user_pos.get_loc(user_id)
    top = metrics.top_k(item_cf.explicit_scores(R_dev, R_ind, user_mean, S, [u]), R_train[[u]], N)[0]
    return books.set_index('id').loc[idx.books[top], ['title', 'authors']]

example_user = test.user_id.value_counts().index[0]
recommend_item_cf(example_user)

In [ ]:
row = test[test.user_id == example_user].iloc[0]
u   = idx.user_pos.get_loc(example_user)
print(f'Пользователь {example_user}: оценок в train {n_rated[u]}, средняя {user_mean[u]:.2f}')
print(f'Книга: {books.set_index("id").loc[row.book_id, "title"]}')
print(f'Предсказание: {predict_rating(example_user, row.book_id):.2f}, фактическая оценка: {row.rating:.0f}')

Чтобы проверить, как работает предсказание оценок, оно считается для всех пар из теста. Предсказания считаются блоками пользователей: для блока одно умножение разреженных матриц даёт оценки по всем книгам сразу. Отдельно считается покрытие: доля тестовых пар, у которых среди соседей книги нашлась хотя бы одна книга, оценённая пользователем. Для пар без соседей прогноз совпадает со средней оценкой пользователя.

In [ ]:
u_test = idx.user_pos.get_indexer(test.user_id)
b_test = idx.book_pos.get_indexer(test.book_id)

rows_k = []
for k in (20, 50, 200):
    S_k           = item_cf.item_similarity(R_dev, R_ind, k_neighbors=k)
    pred, support = item_cf.predict_pairs(R_dev, R_ind, user_mean, S_k, u_test, b_test)
    rows_k.append({'k соседей': k, 'RMSE': metrics.rmse(pred, test.rating), 'покрытие': (support > 0).mean()})

baseline = user_mean[u_test]
rows_k.append({'k соседей': 'средняя пользователя', 'RMSE': metrics.rmse(baseline, test.rating), 'покрытие': np.nan})
pd.DataFrame(rows_k)

In [ ]:
pred, support = item_cf.predict_pairs(R_dev, R_ind, user_mean, S, u_test, b_test)
activity      = pd.cut(n_rated[u_test], bins=[0, 10, 30, 100, 1000], labels=['4–10', '11–30', '31–100', '>100'])
by_activity   = (pd.DataFrame({'группа': activity, 'покрыто': support > 0,
                               'err_cf': (pred - test.rating.values) ** 2,
                               'err_mean': (baseline - test.rating.values) ** 2})
                 .groupby('группа', observed=True)
                 .agg(пар=('покрыто', 'size'), покрытие=('покрыто', 'mean'),
                      RMSE_item_cf=('err_cf', lambda x: np.sqrt(x.mean())),
                      RMSE_средняя=('err_mean', lambda x: np.sqrt(x.mean()))))
by_activity.index.name = 'оценок в train'
by_activity

In [ ]:
print(f'Доля тестовых пар от пользователей с > 100 оценками в train: {(activity == ">100").mean():.1%}')
print(f'Выигрыш RMSE item-based CF над средней пользователя: {1 - metrics.rmse(pred, test.rating) / metrics.rmse(baseline, test.rating):.1%}')

Item-based CF снижает RMSE с 0,868 (средняя оценка пользователя) до 0,837, на 3,6%. Число соседей почти не влияет на ошибку: при росте k с 20 до 200 покрытие растёт с 67% до 89%, но добавленные соседи слабые, и единица в знаменателе почти не даёт им сдвинуть прогноз. Дальше используется k = 50.

Разбивка по активности показывает разреженность в действии. У пользователей с 4–10 оценками в train только у 47% тестовых пар среди 50 соседей книги нашлась хотя бы одна прочитанная пользователем книга, и выигрыш над средней пользователя около 1%. У пользователей со 100+ оценками покрытие 97%, и выигрыш около 6%. Метод работает тем лучше, чем больше история пользователя, а для большинства пользователей она короткая.

Общее покрытие кажется высоким, потому что тест собран из пользователей с 5+ оценками, а активные пользователи дают больше тестовых пар: 11,8% тестовых пар принадлежат пользователям со 100+ оценками в train.

## 13. Вычислительная сложность item-based CF

Наивный подсчёт сходства всех пар книг по плотным векторам пользователей стоит O(n_books² · n_users): 10⁸ пар книг на 53 тыс. пользователей, около 5·10¹² операций. Полная матрица сходства хранит 10⁸ чисел, это 0,8 ГБ в float64.

Разреженное умножение Cᵀ·C обходит только пары книг с общим читателем, и работа пропорциональна Σ n_u², сумме квадратов числа оценок пользователей. В train это 3,6·10⁷, в 140 тысяч раз меньше наивного варианта. Матрица считается блоками по 1000 книг, поэтому в памяти одновременно 1000 × 10 000 значений, а после отбора 50 соседей на книгу остаётся 5·10⁵ чисел вместо 10⁸. Предсказание для пользователя стоит O(n_u · k): каждая его оценка вносит вклад в k соседей.

Квадрат в Σ n_u² делает главными потребителями ресурсов самых активных пользователей: 10% самых активных дают 78% этой суммы. Отсюда способы оптимизации для больших данных. Сходства книг меняются медленно, поэтому их пересчитывают офлайн раз в сутки, а онлайн остаётся одно разреженное умножение. Хранится только топ-k соседей на книгу. Историю сверхактивных пользователей ограничивают или сэмплируют, так как она стоит квадратично. Пары с малым числом общих читателей отбрасываются порогом. При миллионах книг блоки распределяются по кластеру (Spark), а при переходе к векторным представлениям книг соседей ищут приближённым поиском (HNSW, FAISS).

## 14. Матричное разложение: SVD

SVD из библиотеки surprise это не классическое сингулярное разложение, а факторизация со смещениями в духе Саймона Фанка: r̂ = μ + b_u + b_i + q_iᵀp_u, где μ общая средняя оценка, b_u и b_i смещения пользователя и книги, p_u и q_i их скрытые векторы. Модель обучается стохастическим градиентным спуском только по известным оценкам и минимизирует Σ (r − r̂)² + λ(b_u² + b_i² + ‖p_u‖² + ‖q_i‖²). Пустые клетки матрицы в обучении не участвуют, поэтому их не нужно заполнять, как пришлось бы для классического SVD.

Используется то же разделение train/test, что и для item-based CF. Для сравнения обучается BaselineOnly из той же библиотеки: модель только со смещениями μ + b_u + b_i, без скрытых факторов. Разница между ними показывает, сколько дают сами факторы.

In [ ]:
!pip install -q scikit-surprise

In [ ]:
from surprise import SVD, BaselineOnly
from surprise.model_selection import GridSearchCV
from recsys.svd import to_surprise, SVDScorer

train_data, trainset, testset = to_surprise(train, test)
print(f'trainset: {trainset.n_users:,} пользователей, {trainset.n_items:,} книг; testset: {len(testset):,} пар')

Число факторов и сила регуляризации подбираются кросс-валидацией на 3 фолдах внутри train, тестовая выборка в подборе не участвует. Сетка небольшая: 4 комбинации, 12 обучений.

In [ ]:
param_grid = {'n_factors': [50, 100], 'reg_all': [0.02, 0.05],
              'n_epochs': [20], 'lr_all': [0.005], 'random_state': [SEED]}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(train_data)

cv_table = (pd.DataFrame(gs.cv_results)[['param_n_factors', 'param_reg_all', 'mean_test_rmse', 'mean_fit_time']]
            .sort_values('mean_test_rmse'))
cv_table

In [ ]:
best_params = gs.best_params['rmse']
print(f'Лучшие параметры: {best_params}')

svd        = SVD(**best_params).fit(trainset)
bias_model = BaselineOnly(verbose=False).fit(trainset)

svd_test_pred  = np.array([p.est for p in svd.test(testset)])
bias_test_pred = np.array([p.est for p in bias_model.test(testset)])

rmse_table = pd.DataFrame({
    'модель': ['средняя пользователя', 'item-based CF', 'BaselineOnly', 'SVD'],
    'RMSE':   [metrics.rmse(baseline, test.rating), metrics.rmse(pred, test.rating),
               metrics.rmse(bias_test_pred, test.rating), metrics.rmse(svd_test_pred, test.rating)],
})
rmse_table

In [ ]:
by_activity['RMSE_svd'] = (pd.DataFrame({'группа': activity, 'err': (svd_test_pred - test.rating.values) ** 2})
                           .groupby('группа', observed=True).err.mean().pow(0.5).values)
by_activity

Кросс-валидация выбрала 50 факторов и более сильную регуляризацию (0,05): на этих данных больше параметров вредит, у половины пользователей не больше 10 оценок, и лишние факторы подстраиваются под шум.

SVD даёт лучший RMSE на тесте: 0,831 против 0,837 у item-based CF и 0,868 у средней пользователя. Самое заметное в таблице другое: BaselineOnly, у которого нет скрытых факторов, а только общая средняя и смещения пользователя и книги, даёт 0,839, почти как item-based CF. Большую часть предсказуемого в оценках объясняют два числа: насколько щедр читатель и насколько книга нравится всем. Скрытые факторы добавляют к этому меньше 1%.

По группам активности модели расходятся. Для пользователей с 4–10 оценками SVD заметно лучше item-based CF (0,875 против 0,915): там, где у книги нет соседей среди прочитанного, item-based откатывается к средней пользователя, а SVD всё равно знает смещение книги. Для активных пользователей картина обратная: при 31–100 оценках модели почти равны, а при 100+ оценках item-based CF лучше (0,836 против 0,855), потому что длинная история даёт много прямых соседей, и они точнее сжатого 50-мерного профиля.

Для рекомендаций прогноз считается сразу по всем книгам из параметров модели: μ + b_u + b_i + Q·p_u даёт вектор оценок одним матричным умножением, без вызова predict для каждой книги. Ячейка ниже сверяет такой прогноз с predict из surprise на нескольких тестовых парах.

In [ ]:
svd_block = SVDScorer(svd, trainset, idx)

def get_recommendations(user_id, N=5):
    u   = idx.user_pos.get_loc(user_id)
    sc  = svd_block([u])
    top = metrics.top_k(sc.copy(), R_train[[u]], N)[0]
    return (pd.Series(sc[0, top], index=idx.books[top], name='прогноз')
            .to_frame().join(books.set_index('id')[['title', 'authors']]))

check  = test.sample(5, random_state=SEED)
manual = [svd_block([idx.user_pos.get_loc(u)])[0, idx.book_pos.get_loc(b)] for u, b in zip(check.user_id, check.book_id)]
native = [svd.predict(u, b, clip=False).est for u, b in zip(check.user_id, check.book_id)]
print(f'Ручной прогноз совпадает с predict: {np.allclose(manual, native)}')

In [ ]:
get_recommendations(example_user, N=5)

Рекомендации SVD для того же пользователя состоят из пяти сборников Calvin and Hobbes, то есть из той же группы книг, что занимает верх неперсонализированного топа в разделе 9. Модель обучена минимизировать ошибку оценки, и самый надёжный способ предсказать высокую оценку это взять книги с самым большим смещением b_i, которые нравятся почти всем, кто их открыл. Смещение книги одинаково для всех пользователей, поэтому оно тянет персональный список к общему топу.

Это расхождение между задачей предсказания оценки и задачей ранжирования: модель с лучшим RMSE не обязана давать лучший топ-N. Проверить это должны метрики ранжирования на следующем этапе.

## 15. Оценка качества рекомендаций

Протокол одинаков для всех моделей. Каждая модель ранжирует все книги каталога, кроме тех, что пользователь оценил в train, и отдаёт топ-K. Релевантными считаются книги из теста пользователя с оценкой не ниже порога. Основной порог 4, как в задании. Так как две трети оценок не ниже 4 (раздел 3), дополнительно считается порог 5, при котором релевантна только самая любимая треть. В оценку входят тестовые пользователи, у которых есть хотя бы одна релевантная книга.

Precision@K это доля релевантных книг в топ-K, Recall@K это доля релевантных книг пользователя, попавших в топ-K. nDCG@K учитывает порядок: попадание на позиции i весит 1 / log₂(i + 1). Идеальный DCG строится из всех релевантных тестовых книг пользователя (не больше K штук), а не из выданного списка, иначе модель, нашедшая одну релевантную книгу из десяти, получила бы максимум.

Неоценённая книга в топе считается нерелевантной, хотя пользователь, возможно, просто её не читал. Поэтому абсолютные значения метрик занижены, и сравнивать модели имеет смысл только между собой.

Помимо точности считаются две характеристики выдачи: покрытие каталога (доля книг, попавших хотя бы в один топ-10) и медианный ratings_count рекомендованных книг на Goodreads. Вторая показывает, насколько модель тянется к известным книгам. Для нижней границы добавлена случайная модель.

In [ ]:
K_LIST        = (5, 10)
gr_popularity = books.set_index('id').loc[idx.books, 'ratings_count'].values

rel_4 = metrics.relevance_matrix(test, idx, 4)
rel_5 = metrics.relevance_matrix(test, idx, 5)

def evaluate(score_fn, rel):
    return metrics.evaluate(score_fn, rel, R_train, gr_popularity, k_list=K_LIST)

n_eval_4, n_eval_5 = np.diff(rel_4.indptr).astype(bool).sum(), np.diff(rel_5.indptr).astype(bool).sum()
print(f'Пользователей в оценке: порог 4: {n_eval_4:,}, порог 5: {n_eval_5:,}')
print(f'Релевантных книг на пользователя: порог 4: {rel_4.nnz / n_eval_4:.2f}, порог 5: {rel_5.nnz / n_eval_5:.2f}')

Функции оценок возвращают для блока пользователей матрицу прогнозов по всем книгам. Популярная модель обучается заново на train и даёт всем один и тот же вектор, item-based CF и SVD считают блок одним матричным умножением.

In [ ]:
pop_vec = popularity.popularity_vector(popularity.popularity_scores(train), idx.book_pos)

models = {
    'Random':        lambda u: np.random.default_rng(SEED + int(u[0])).random((len(u), len(idx.books))),
    'Popularity':    lambda u: np.tile(pop_vec, (len(u), 1)),
    'Item-based CF': lambda u: item_cf.explicit_scores(R_dev, R_ind, user_mean, S, u),
    'SVD':           svd_block,
}

results = pd.DataFrame([{'порог': threshold, 'модель': name, **evaluate(fn, rel)}
                        for threshold, rel in ((4, rel_4), (5, rel_5)) for name, fn in models.items()])
results.round(4)

In [ ]:
def format_results(df):
    out = df.rename(columns={'покрытие каталога': 'покрытие', 'медиана ratings_count': 'ratings_count'})
    for col in out.columns[2:]:
        out[col] = out[col].map(lambda v: f'{v:,.0f}' if col == 'ratings_count' else f'{v:.4f}')
    return out

show_table(format_results(results), 'Сравнение моделей на тестовой выборке (ratings_count: медиана по рекомендованным книгам)'); plt.show()

In [ ]:
main = results[results['порог'] == 4].set_index('модель')
x    = np.arange(len(main))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ('nDCG@10', 'Recall@10')):
    ax.bar(x - 0.2, main[metric], width=0.4, color='steelblue', label='порог 4')
    ax.bar(x + 0.2, results[results['порог'] == 5].set_index('модель').loc[main.index, metric],
           width=0.4, color='coral', label='порог 5')
    ax.set_xticks(x)
    ax.set_xticklabels(main.index)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} по моделям')
    ax.legend()
plt.tight_layout(); plt.show()

Item-based CF единственная модель, заметно лучше случайной: nDCG@10 0,147 при пороге 4 и 0,179 при пороге 5, на два порядка выше случайной выдачи. Популярная модель и SVD остаются на уровне случайной: nDCG@10 0,0009 и 0,0010.

У SVD лучший RMSE (раздел 14), но в ранжировании он проигрывает item-based CF больше чем в сто раз. Причина видна в двух последних колонках таблицы. SVD рекомендует всего 1,1% каталога, и медианная популярность его рекомендаций 17 179, ровно как у популярной модели: топ-10 почти любого пользователя состоит из книг с наибольшим смещением b_i, тех же сборников, что в неперсонализированном топе. Чтобы оценка была близка к истинной, модели выгодно опираться на смещения, а персональные факторы меняют прогноз на доли балла и не перетасовывают верх списка.

Хорошо предсказать оценку и хорошо выбрать, что человек прочитает, это разные задачи. RMSE считается только по книгам, которые пользователь уже выбрал сам. Ранжирование идёт по всему каталогу, и там важнее всего угадать сам выбор. Item-based CF рекомендует книги, похожие на прочитанные, и поэтому попадает в то, что человек выберет дальше. Его покрытие 99% каталога, а медианная популярность рекомендаций (21 829) близка к медиане каталога, то есть модель не тянется к хитам.

Порог 5 не меняет порядок моделей. Recall и nDCG при нём выше, потому что релевантных книг меньше (2,65 на пользователя против 3,75), и найти заметную их долю проще. Выводы устойчивы к выбору порога.

## 16. Item-based CF на неявном сигнале

SVD показал, что точное предсказание оценки не даёт хорошего ранжирования: чтобы книга оказалась релевантной, пользователь сначала должен её выбрать и прочитать. Сам выбор лучше описывает неявный сигнал: прочитал или нет. Ниже item-based CF строится на бинарной матрице: сходство книг считается косинусом векторов «кто прочитал» с тем же сжатием по числу общих читателей, а оценка книги для пользователя равна сумме её сходств с прочитанными им книгами. Предсказывается уже не оценка, а то, что пользователь выберет следующим.

Для сравнения в ту же таблицу добавлена контентная модель: профиль пользователя это средний TF-IDF-вектор прочитанных книг, оценка книги это косинусная близость её вектора к профилю.

In [ ]:
S_bin = item_cf.item_similarity(R_ind, R_ind, k_neighbors=50, shrink=10)
X_n   = normalize(X)

implicit_block = lambda u: item_cf.implicit_scores(R_ind, S_bin, u)
content_block  = lambda u: content.user_profile_scores(R_ind[u], X_n)

extra = [{'порог': threshold, 'модель': name, **evaluate(fn, rel)}
         for threshold, rel in ((4, rel_4), (5, rel_5))
         for name, fn in (('Item-based CF (неявный)', implicit_block), ('Content-based', content_block))]

results_all = (pd.concat([results, pd.DataFrame(extra)])
               .sort_values(['порог', 'nDCG@10']).reset_index(drop=True))
results_all.round(4)

In [ ]:
show_table(format_results(results_all), 'Итоговое сравнение моделей (ratings_count: медиана по рекомендованным книгам)'); plt.show()

Item-based CF на неявном сигнале лучший по всем метрикам точности: nDCG@10 0,216 против 0,147 у версии на явных оценках при пороге 4, Recall@10 0,256 против 0,162. Разница меньше при пороге 5 (0,193 против 0,179): среди самых любимых книг оценка несёт больше информации, но и там бинарная версия впереди. Отклонения оценок от среднего мешают ранжированию: книга, которую пользователь прочитал и оценил ниже своей средней, в явной версии отталкивает похожие книги, хотя сам факт прочтения говорит, что такие книги он выбирает.

Контентная модель с nDCG@10 0,109 уступает обоим вариантам CF, но на два порядка лучше популярной модели и SVD. При этом она рекомендует самые известные книги среди всех моделей (медианный ratings_count 25 533): у популярных книг больше тегов, и их профили богаче.

## 17. Гибрид для холодного старта

Холодный старт бывает двух видов, и для них нужны разные решения.

Новый пользователь. Предлагается взвешенный гибрид, в котором вес коллаборативной части растёт с длиной истории: score = α · z(CF) + (1 − α) · z(content), α = n / (n + n₀), где n число прочитанных книг, а z означает стандартизацию оценок внутри пользователя, чтобы шкалы двух моделей были сравнимы. При n₀ = 10 у пользователя с 2 книгами вес CF 0,17, с 30 книгами 0,75. Параметр n₀ зафиксирован заранее, без подбора на тесте: для честного подбора нужна отдельная валидационная выборка.

Новая книга. У книги без оценок нет ни одного соседа, и коллаборативная модель не может её рекомендовать никому. Проверка: 500 случайных книг полностью удаляются из train, как будто они только что появились в каталоге. Для них отдельный блок «Новинки» ранжирует только новые книги, и сравнивается, кто лучше угадывает, какие из них пользователь прочитает и оценит не ниже 4: случайный порядок, CF или контентная модель.

In [ ]:
N0 = 10

def zrow(a):
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-12)

def hybrid_block(u):
    alpha = (n_rated[u] / (n_rated[u] + N0))[:, None]
    return alpha * zrow(implicit_block(u)) + (1 - alpha) * zrow(content_block(u))

per_user = {}
for name, fn in (('Item-based CF (неявный)', implicit_block), ('Content-based', content_block), ('Гибрид', hybrid_block)):
    users_eval, per_user[name] = metrics.ndcg_per_user(fn, rel_4, R_train)

hybrid_table = pd.DataFrame(per_user)
hybrid_table['оценок в train'] = pd.cut(n_rated[users_eval], bins=[0, 10, 30, 100, 1000],
                                        labels=['4–10', '11–30', '31–100', '>100'])
hybrid_by_group = hybrid_table.groupby('оценок в train', observed=True).mean()
hybrid_by_group.loc['все'] = hybrid_table.drop(columns='оценок в train').mean()
hybrid_by_group.round(4)

Гибрид не улучшает рекомендации для пользователей из теста ни в одной группе: даже при 4–10 оценках nDCG@10 гибрида 0,206 против 0,208 у чистого CF. Контентная модель сильнее всего именно у малоактивных пользователей (0,120 против 0,084 у пользователей со 100+ оценками), но коллаборативный сигнал уже при 4 прочитанных книгах точнее. Это честный отрицательный результат: на этих данных подмешивать контент для пользователей с историей не нужно.

Для пользователей совсем без истории ни CF, ни контентная модель не работают, обеим нужна хотя бы одна прочитанная книга. Такие пользователи в тест не попали (порог 5 оценок), и для них остаётся неперсонализированный топ или короткий опрос о любимых книгах при регистрации, после которого включается контентная модель.

In [ ]:
rng_cold   = np.random.default_rng(SEED)
cold_books = rng_cold.choice(len(idx.books), 500, replace=False)
warm       = np.ones(len(idx.books), dtype=bool)
warm[cold_books] = False

R_ind_cold = (R_ind @ sparse.diags(warm.astype(float))).tocsr()
R_ind_cold.eliminate_zeros()
S_cold     = item_cf.item_similarity(R_ind_cold, R_ind_cold, k_neighbors=50)
rel_cold   = (rel_4 @ sparse.diags((~warm).astype(float))).tocsr()
rel_cold.eliminate_zeros()

def new_books_slot(score_fn):
    def block(u):
        scores = score_fn(u)
        scores[:, warm] = -np.inf
        return scores
    return block

cold_models = {
    'Random':        models['Random'],
    'Item-based CF': lambda u: item_cf.implicit_scores(R_ind_cold, S_cold, u),
    'Content-based': lambda u: content.user_profile_scores(R_ind_cold[u], X_n),
}
cold_table = pd.DataFrame([{'модель': name, **evaluate(new_books_slot(fn), rel_cold)} for name, fn in cold_models.items()])

print(f'Новых книг: {len(cold_books)}, пар пользователь–новая книга в тесте с оценкой >= 4: {rel_cold.nnz:,}')
cold_table[['модель', 'Recall@10', 'nDCG@10']].round(4)

Для новых книг картина обратная. CF ранжирует их не лучше случайного порядка (Recall@10 около 0,02 против 0,022): у книги без оценок нет соседей, и все её оценки равны нулю. Контентная модель находит 44,6% новых книг, которые пользователь затем прочитал и оценил не ниже 4, в 20 раз больше случайного порядка. Её профиль строится из названия и тегов и не требует ни одной оценки.

Отсюда предлагаемая схема гибрида. Основная лента строится item-based CF на неявном сигнале. Новые книги получают отдельный блок «Новинки», ранжированный контентной моделью, пока у книги не наберётся достаточно читателей для коллаборативного сходства. Пользователям без истории показывается топ по известности (ratings_count на Goodreads), а после первых прочитанных книг включается CF.

## 18. Ограничения офлайн-оценки

Высокое качество контентной модели и CF требует проверки: какие именно книги они угадывают. Ниже для каждой модели считается доля рекомендаций и доля попаданий, у которых первый автор совпадает с автором хотя бы одной книги из истории пользователя.

In [ ]:
first_author = pd.factorize(books.set_index('id').loc[idx.books, 'authors'].str.split(',').str[0].str.strip())[0]

author_table = pd.DataFrame({name: metrics.same_author_share(fn, rel_4, R_train, first_author)
                             for name, fn in (('Item-based CF', models['Item-based CF']),
                                              ('Item-based CF (неявный)', implicit_block),
                                              ('Content-based', content_block))}).T
author_table.round(3)

86–90% попаданий у всех трёх моделей приходятся на книги авторов, которых пользователь уже читал: продолжения серий и другие книги того же автора. У CF на неявном сигнале две трети всех рекомендаций относятся к уже знакомым авторам. Офлайн-метрики вознаграждают модель за предсказание, что после первой книги серии человек прочитает вторую, а такую рекомендацию он нашёл бы и без сервиса.

Поэтому абсолютные значения метрик завышают реальную пользу рекомендаций, а преимущество одной модели над другой частично объясняется тем, насколько охотно она предлагает продолжения. Реальный вклад рекомендаций показал бы только онлайн-эксперимент: A/B-тест с выручкой или числом прочитанных книг, где учитываются и книги, которые пользователь нашёл бы сам.

## 19. Выводы

Лучшее качество ранжирования показал item-based CF на неявном сигнале: nDCG@10 0,216, Recall@10 0,256 при пороге релевантности 4, и он остаётся лучшим при пороге 5. Он описывает то, что определяет релевантность в этой задаче: какие книги пользователь выбирает читать. Сходство книг по совместным прочтениям устойчиво, потому что у каждой книги около 100 читателей, а модель покрывает 90% каталога и не тянется к хитам.

Сильные и слабые стороны моделей. Популярная модель по средней оценке проста и не требует истории, но на этих данных бесполезна: высокую среднюю получают сборники для поклонников, которые мало кто читает, и её nDCG@10 на уровне случайной выдачи. Контентная модель работает без оценок и единственная умеет рекомендовать новые книги (Recall@10 0,446 в блоке новинок против 0,02 у CF), но повторяет авторов и жанры из истории и тянется к известным книгам. Item-based CF на явных оценках хорошо предсказывает оценку активным пользователям (RMSE 0,836 при 100+ оценках) и объясним через соседние книги, но центрирование оценок ухудшает ранжирование, а у малоактивных пользователей половина прогнозов остаётся без опоры. SVD даёт лучший RMSE (0,831), но почти весь выигрыш дают смещения пользователя и книги, и в ранжировании он не отличается от популярной модели: nDCG@10 0,0010 и покрытие 1,1% каталога.

Главный урок работы: модель, лучше всех предсказывающая оценку, может хуже всех ранжировать. RMSE считается по книгам, которые пользователь уже выбрал, а рекомендация требует угадать сам выбор.

Пути улучшения. Первый шаг это модели, обученные на ранжирование и неявный сигнал, например implicit ALS или BPR вместо SVD на оценках: они исправляют ровно ту проблему, которую показал SVD. Гибрид из раздела 17 стоит собрать в рабочую схему: CF для основной ленты, контентная модель для новинок, известность по ratings_count для пользователей без истории.

Повторы авторов и серий лучше убрать из основной ленты или вынести в отдельный блок «Продолжить серию», а качество дополнительно мерить новизной и разнообразием выдачи. Контентные признаки можно обогатить эмбеддингами описаний (BERT) вместо TF-IDF по тегам, а файл to_read.csv с отложенными книгами даёт ещё один неявный сигнал, который сейчас не используется. При росте каталога естественна двухэтапная схема: несколько моделей генерируют кандидатов, бустинг с признаками пользователя и книги их ранжирует.

Окончательную проверку даёт только онлайн-эксперимент, потому что офлайн-метрики на этих данных в основном измеряют угадывание продолжений.